## Imports

In [ ]:
import datetime
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import src.XRO
import copy
import scipy.stats
import warnings
import calendar
import pandas as pd

# import gsw

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Funcs

In [ ]:
def get_Q(data, fn):
    """get heat flux over region using specified fn"""

    ### reconstruct flux over region
    hf = src.utils.reconstruct_wrapper(data, fn=fn)

    ## convert units to K/yr
    sec_per_mo = 8.64e4 * 30
    sec_per_yr = 12 * sec_per_mo
    rho = 1.02e3
    Cp = 4.2e3
    H = 50
    # H = 80

    ## apply scaling fac.
    Q = hf * sec_per_yr / (rho * Cp * H)

    return Q

def sel_eq(data):
    """select equatorial region"""
    idx = dict(latitude=slice(-5,5), longitude=slice(120,280))
    return data.sel(idx).mean("latitude")

def sel_epac(data):
    """select equatorial region"""
    idx = dict(longitude=slice(210,270))
    return data.sel(idx).mean("longitude")

## Load data

### surface heat fluxes

In [ ]:
flux_forced, flux_anom = src.utils.load_flux_data()

## switch sign convection to be positive into the ocean
for v in ["flns", "lhflx"]:
    flux_anom[v] = -1 * flux_anom[v]
    flux_forced[v] = -1 * flux_forced[v]

## load into memory
flux_anom.load();

### $T$, $h$

In [ ]:
Th = src.utils.load_cesm_indices().compute()

In [ ]:
T = xr.merge(
    [(Th["T_3"]**i).rename(f"T{i}") for i in range(4)]
)

### merge

In [ ]:
data = xr.merge(
    [
        T, 
        flux_anom[[v for v in list(flux_anom) if "comp" not in v]]
    ]
)

## window in time
get_windowed = lambda x : src.utils.get_windowed(x, stride=120, window_size=480)
data = get_windowed(data)
data = data.sel(year=slice(None,2080))

## fit

In [ ]:
def regress_pinv(X, x_vars, y_var):
    """do nonlinear regression"""

    ## prep data
    # prep = lambda x: x.transpose("member", ...)
    prep = lambda x: x.stack(s=["time", "member"]).transpose("year",..., "s")

    X_ = prep(X[x_vars].to_dataarray(dim="v"))
    Y_ = prep(X[y_var]).transpose("year","mode",...)

    ## empty array to hold results
    m = xr.DataArray(
        coords={"year": X.year, "mode": Y_.mode, "v": x_vars},
        dims=["year", "mode", "v"],
    )

    ## do regression
    X_pinv = np.linalg.pinv(X_.values)
    m.values = np.einsum("...ki,...ij->...kj", Y_.values, X_pinv)

    return m.rename({"v": "param"})


def regress_pinv_wrapper(X, x_vars, y_var):
    return X.groupby("time.month").map(regress_pinv, x_vars=x_vars, y_var=y_var)

In [ ]:
flux_types=["fsns", "lhflx", "flns"]

## empty array to hold results
alpha = []

## loop thru flux types
for ftype in flux_types:

    kwargs = dict(x_vars=["T0", "T1", "T2", "T3"], y_var=ftype)
    alpha.append(-regress_pinv_wrapper(data, **kwargs))

## merge
alpha = xr.merge([a.rename(f) for a, f in zip(alpha, flux_types)])

## add components
alpha = xr.merge([alpha, flux_anom[[f"{f}_comp" for f in flux_types]]])

## compute to heat flux
Q = get_Q(alpha, fn=sel_eq)
Q_epac = get_Q(alpha, fn=sel_epac)

## plot results

### timeseries

In [ ]:
sel_ = lambda x : x.sel(longitude=slice(210,270), month=slice(6,12)).mean(["longitude","month"])
delta = lambda x : sel_(x)-sel_(x).isel(year=0)

fig,axs = plt.subplots(1,4,figsize=(12,3), layout="constrained")
for ax, p in zip(axs, list(Q.param)):
    for f in flux_types:
        ax.plot(Q.year, delta(Q[f].sel(param=p)), label=f)

    ax.axhline(0, ls="--", c="k", lw=1)
    ax.legend()

src.utils.set_lims(axs)
plt.show()

### Plot spatial structure

In [ ]:
sel_ = lambda x : x.sel(month=slice(6,12)).mean(["month"])

fig,axs = plt.subplots(1,4,figsize=(10,2.5), layout="constrained")

for ax, p in zip(axs, Q.param.values):

    for y in [1870, 2000, 2080]:

        ax.plot(Q.longitude, sel_(Q["fsns"]).sel(year=y, param=p))
        
    ax.set_yticks([])
    ax.axhline(0, ls="--", c="k", lw=.8)
    ax.set_title(p)

src.utils.set_lims(axs)
src.utils.add_vticks(axs, xticks=[210,270], xlines=[210,270])
plt.show()


In [ ]:
sel_ = lambda x : x.sel(month=slice(6,12)).mean(["month"])

fig,axs = plt.subplots(1,3,figsize=(7.5,2.5), layout="constrained")

# for ax, p in zip(axs, Q.param.values):

for ax, y in zip(axs,[1870, 2000, 2080]):

    ax.plot(Q.longitude, sel_(Q["fsns"]).sel(year=y, param="T0"))
    ax.plot(Q.longitude, -sel_(Q["fsns"]).sel(year=y, param="T2"))
        
    ax.set_yticks([])
    ax.axhline(0, ls="--", c="k", lw=.8)
    ax.set_title(y)

src.utils.set_lims(axs)
src.utils.add_vticks(axs, xticks=[210,270], xlines=[210,270])
plt.show()


In [ ]:
sel_ = lambda x : x.sel(month=slice(6,12)).mean(["month"])

fig,axs = plt.subplots(1,4,figsize=(10,2.5), layout="constrained")

for ax, p in zip(axs, Q.param.values):

    for y in [1870, 2000]:

        ax.plot(Q_epac.latitude, sel_(Q_epac["fsns"]).sel(year=y, param=p))
        
    ax.set_yticks([])
    ax.axhline(0, ls="--", c="k", lw=.8)
    ax.set_title(p)

src.utils.set_lims(axs)
src.utils.add_vticks(axs, xticks=[-5,5], xlines=[-5,5])
plt.show()


### Plot climatology
Could also plot rainfall

In [ ]:
fsns_forced = get_Q(get_windowed(flux_forced[["fsns","fsns_comp"]]), fn=sel_eq)["fsns"]
fsns_clim = fsns_forced.groupby("time.month").mean()

In [ ]:
fig, ax = plt.subplots(figsize=(4,3), layout="constrained")

for y in [1870, 2000]:

    ax.plot(Q.longitude, sel_(fsns_clim).sel(year=y))

# ax.axhline(0, ls="--", c="k", lw=.8)

src.utils.add_vticks([ax], xticks=[210,270], xlines=[210,270])
plt.show()


#### Load cloud fraction

In [ ]:
cld_forced, cld_anom = src.utils.load_cloud_data()

In [ ]:
cld_comp, cld_forced_scores = src.utils.split_components(cld_forced)
_, cld_anom_scores = src.utils.split_components(cld_anom)

In [ ]:
cld_total_scores = cld_forced_scores + cld_anom_scores
cld_total = xr.merge(
    [cld_total_scores, cld_comp.rename({v:f"{v}_comp" for v in list(cld_comp)})]
)

In [ ]:
cld_clim = get_windowed(cld_forced).groupby("time.month").mean()
cld_clim_eq = src.utils.reconstruct_wrapper(cld_clim, fn=sel_eq)

In [ ]:
sel_ = lambda x : x.sel(month=slice(6,12)).mean(["month"])

fig,axs = plt.subplots(1,3,figsize=(7.5,2.5), layout="constrained")

for ax, v in zip(axs, list(cld_clim_eq)):

    for y in [1870, 2010, 2080]:

        ax.plot(cld_clim_eq.longitude, sel_(cld_clim_eq[v]).sel(year=y), label=y)
        
    ax.set_yticks([])
    ax.axhline(0, ls="--", c="k", lw=.8)
    ax.set_title(v[-3:])


axs[0].set_yticks([0, 0.5, 1])
axs[0].set_ylabel("Cloud fraction")
axs[-1].legend()
src.utils.set_lims(axs)
src.utils.add_vticks(axs, xticks=[210,270], xlines=[210,270])
plt.show()


Timeseries

In [ ]:
Q_cld = sel_(cld_clim_eq.sel(longitude=slice(210,270)).mean("longitude"))

In [ ]:
delt = lambda x : x-x.isel(year=0)

fig,ax = plt.subplots(figsize=(4,3))

for v in list(Q_cld):
    ax.plot(Q_cld.year, delt(Q_cld[v]), label=v)

## label/formatting
ax.axhline(0, ls="--", c="k", lw=.8)
ax.legend()

plt.show()

#### Scatter

In [ ]:
Q_cld_tot = src.utils.reconstruct_wrapper(
    cld_total, fn=src.utils.get_nino3,
)

## add Q data and window
Th_ = get_windowed(xr.merge([Th, Q_cld_tot]))

In [ ]:
sel_ = lambda x : x.sel(time=(x.time.dt.month==12))
fig,axs = plt.subplots(3,3,figsize=(8,8))

for j, y in enumerate([1870, 2000, 2080]):

    axs_ = axs[j]

    for ax, v in zip(axs_,list(Q_cld_tot)):
        ax.scatter(sel_(Th_["T_3"].sel(year=y)), sel_(Th_[v].sel(year=y)), s=7)

        if j==0:
            ax.set_title(v[-3:])
    
        ax_kwargs = dict(ls="--", c="k", lw=1)
        ax.axhline(**ax_kwargs)
        ax.axvline(**ax_kwargs)
        if j<2:
            ax.set_xticks([])
        else:
            ax.set_xticks([-3,0,3])
            ax.set_xlabel(r"$T_3$")
    axs_[0].set_ylabel("Fraction")

src.utils.set_lims(axs)

plt.show()

### Get fits over time